In [13]:
import os
import polars as pl

In [14]:
script_path = os.getcwd()
project_path = os.path.join(script_path, '..', '..')
features_dir = os.path.join(project_path, 'data', 'features')
feature_file_path = os.path.join(features_dir, 'content_relevance_score.parquet')
feature_df = pl.read_parquet(feature_file_path)

In [15]:
# 1. Mostrar el contenido completo de los strings (sin límite)
pl.Config.set_fmt_str_lengths(1000) # Ajusta al largo que necesites

polars.config.Config

In [16]:
feature_df

comment_id,content_relevance_score,reasoning_content_relevance_score
str,i64,str
"""mzlqxrg""",4,"""The comment expresses a desire for Iran to take military action against Israel in the context of humanitarian aid, which relates directly to the ongoing conflict and its implications for Gaza."""
"""obbcoq3""",0,"""The comment is vague and does not provide any specific information or opinion related to the Gaza conflict, focusing instead on a general statement about truth without context."""
"""n9yjtzg""",2,"""The comment reflects on the hostility faced by Jews in Barcelona, linking it to broader themes of antisemitism and public sentiment related to the Gaza conflict, but it does not directly address the conflict itself or its core events."""
"""nax2wzf""",1,"""The comment focuses on a political strategy regarding the Democratic Party in the context of the Gaza conflict, but it does not directly address the humanitarian situation or the core events of the conflict itself. It is more about U.S. internal politics than the Gaza conflict."""
"""n8vxcji""",3,"""The comment reflects on a historical event related to Israel's actions in Gaza and connects it to current events, indicating a perspective on the conflict's implications. However, it does not engage with the core conflict or humanitarian issues directly."""
…,…,…
"""mqyllfc""",3,"""The comment discusses the demographic composition of Israel and its political implications, which relates to the broader context of the Gaza conflict and the Israeli-Palestinian relations. However, it does not directly address the core conflict or humanitarian issues, making it less relevant."""
"""mraetks""",2,"""The comment expresses a desire for the bakery to fail, which is a reaction to the violent slogans associated with the Israeli bakery. However, it does not engage with the broader implications of the Gaza conflict or the humanitarian situation, making it less relevant to the core issues."""
"""myg8mvn""",0,"""The comment expresses appreciation for the original post but does not engage with the core issues of the Gaza conflict or provide any relevant insights, making it largely off-topic."""


In [17]:
count_relevant_content = sum(feature_df['content_relevance_score'] >= 3) 
content_size = len(feature_df)
prop_relevant_content = count_relevant_content / content_size
print(f"Count relevant content: {count_relevant_content}")
print(f"Percentage relevant content: {round(prop_relevant_content * 100, 2)}%")

Count relevant content: 116850
Percentage relevant content: 49.1%


In [18]:
count_relevant_content = sum(feature_df['content_relevance_score'] >= 4) 
content_size = len(feature_df)
prop_relevant_content = count_relevant_content / content_size
print(f"Count relevant content: {count_relevant_content}")
print(f"Percentage relevant content: {round(prop_relevant_content * 100, 2)}%")

Count relevant content: 86956
Percentage relevant content: 36.54%


In [19]:
feature_df['content_relevance_score'].value_counts().sort('count', descending=True)

content_relevance_score,count
i64,u32
4,55116
0,53832
2,39771
5,31840
3,29894
1,27543
-1,6


In [20]:
processed_data_dir = os.path.join(project_path, 'data', 'processed_data')

base_data_path = os.path.join(processed_data_dir, '02_processed_data.parquet') 

base_df = pl.read_parquet(base_data_path)

processed_df = base_df.join(feature_df, how='left', on='comment_id')

processed_df_filtered = processed_df[['comment_id', 'post_id', 'post_title', 'post_body', 'comment_body', 'content_relevance_score', 'reasoning_content_relevance_score']]

In [21]:
len(processed_df) - processed_df['content_relevance_score'].is_null().sum()

238002

In [22]:
processed_df['content_relevance_score'].is_null().sum()

0

In [23]:
for score in processed_df['content_relevance_score'].unique():
    print(f'content_relevance_score == {score}'.upper(), '\n')
    display(processed_df_filtered.filter(pl.col('content_relevance_score') == score).sample(n=5, seed=123))
    print('='*200, '\n')

CONTENT_RELEVANCE_SCORE == -1 



comment_id,post_id,post_title,post_body,comment_body,content_relevance_score,reasoning_content_relevance_score
str,str,str,str,str,i64,str
"""n8mj7ql""","""1mpr8o4""","""a video of aljazera journalist Anas, who was killed by Israel, talking to his daughter Sham.""","""""","""**Looks like this thread is getting a lot of attention. Greetings, /r/all! Please keep it civil.** *I am a bot, and this action was performed automatically. Please [contact the moderators of this subreddit](/message/compose/?to=/r/Palestine) if you have any questions or concerns.*""",-1,"""The comment is meta-Reddit moderation/attention ("""
"""npf2aoc""","""1ozxpc4""","""At least on Reddit, pro-Palestine subs do not say/discuss 'Globalize the Intifada' much. But pro-Israel subs absolutely do & obsess about it. The 'controversy' over this expression is exaggerated & weaponized to slander critics of Israel as antisemitic.""","""""","""Something, something, projection is the word that is appropriate here.""",-1,"""The comment is a terse, generic rebuttal ("""
"""ocakh06""","""1s2nzvc""","""I hope they realise the truth that they are supporting a gncidl entity.""","""Infront of Elstree and Borehamwood station, London, UK.""","""Oh they realise""",-1,"""Primary subject: accusation of supporting a 'genocidal entity' (core Gaza/Israel conflict) at a London location; the comment ("""
"""n7xwa20""","""1mm4i9x""","""If Pro-Palestinians think ""Zionism"" is an ideology rather than a reference to Jews, then why do they call all Jews who moved to Israel ""Zionists""?""","""I often hear Pro-Palestinian say things like ""I don't hate Jews, I just hate Zionists."" They explain that ""Zionists"" were a political movement to displace Arabs or something, totally separate from the ethnicity and religion of Judaism. If that were true, then why do they call any Jews who moved to Israel ""Zionists""? For instance, they say things like ""Hundreds of thousands of Zionist colonizers immigrated to Israel in the 1800s and 1900s"" even though the majority of Jews who immigrated to Israel were refugees who had no particular political agenda. There were certainly Jews who dreamed of some kind of vague homeland in Israel — originally the dream was to be Ottoman subjects living in the Ottoman empire in Jewish neighborhoods, later when hundreds of groups started dreaming of a nation state, Jews did too — but the reality is, most Jews moved to escape persecution. For instance, in the 1880s, most Jewish immigrants were Russian and Romanian Jewih refugees fled pogroms (violent anti-…","""Couldn't agree more.""",-1,"""The comment is a one-line expression of agreement ("""
"""oaxrhwt""","""1rw7k69""","""Mamdani rips ‘genocide’ in Palestine at St. Patrick’s Day event — after he botches answer on unified Ireland""","""""","""Mayor Shit-eating-grin""",-1,"""The comment is a short insult/nickname ("""



CONTENT_RELEVANCE_SCORE == 0 



comment_id,post_id,post_title,post_body,comment_body,content_relevance_score,reasoning_content_relevance_score
str,str,str,str,str,i64,str
"""n7gnt3d""","""1mk1sih""","""Trump's tariff war: Canada's sovereignty is the real issue, trade expert warns""","""""","""It is utterly insane to suggest that the Canadian public would accept greater integration with the USA, economic or otherwise, in the current context. People are extremely angry with our treacherous former friends. This piece totally misses that""",0,"""The comment focuses on Canadian public opinion regarding integration with the USA, which is unrelated to the Gaza conflict and does not address any relevant topics related to the core conflict."""
"""niuphe3""","""1o38pai""","""What are they going to do now""","""""","""They'll go back to gender pronouns and cancelling people who use the wrong words. You know, as middle school girls do.""",0,"""The comment does not address the Gaza conflict or any related topics, instead focusing on social issues unrelated to the situation in Gaza."""
"""npbw28t""","""1ozjb0c""","""Will the US federal lawsuit filed last month against the PA and PLO result in the end of their use of mass murder of civilians to ""free Palestine?""""","""Palestinian President Mahmoud Abbas, who celebrated his 90th birthday on Saturday and will begin the 22nd year of his first 4-year term in office next year, said in 2015 of the wave of Palestinian inspired mass murders happening at that time in Israel, ""We said to everyone that we want peaceful popular uprising, and that's what this is. That's what this is. However, the aggression of firing bullets has come from the Israelis."" The above from the video of his speech posted earlier today on the palwatch YouTube channel: **Abbas: Murdering Israelis is popular peaceful uprising** Official PA TV, Nov. 16, 2015 Official PA TV News broadcast Excerpt from PA Chairman Mahmoud Abbas' speech at the Palestine Prize ceremony PA Chairman Mahmoud Abbas: ""No one called for this uprising (i.e., Oct. - Nov. 2015 terror campaign) and no one asked for it. It stemmed from the hearts of the young who have seen everything with their own eyes. They have seen the oppression, the settlements and they have seen…","""Hi Dr_G_E, **thank you** for posting in our community! Please check if your post is rule 10 and 11 compliant. Consider deleting immediately before there are comments if it is not, but not after (rule 12). **Reminder to readers:** All comments need to abide by our rules which are designed to maintain constructive discourse. Please review those rules if you are not familiar with them, and remember to report any comments that violate those guidelines. *I am a bot, and this action was performed automatically. Please [contact the moderators of this subreddit](/message/compose/?to=/r/IsraelPalestine) if you have any questions or concerns.*""",0,"""The comment is a bot-generated response that focuses on subreddit rules and moderation, which does not contribute to the discussion about the Gaza conflict or public opinion on it."""
"""ndum7w8""","""1ne0r0x""","""What a clown""","""""","""Glad I never liked Seinfeld now lol""",0,"""The comment does not address the Gaza conflict or any related topics, focusing instead on a personal opinion about a television show, which is irrelevant to the study."""
"""oa7rn8d""","""1rsl9af""","""my prediction for the recolonization of america is coming true.""","""you've all called me crazy and stupid for thinking that america is going to be recolonized by the end of the decade. well, as with so many crazy people in the past, i'm being proven absolutely right. america has fucked with the wrong country. iran has survived and it is hopping mad. and, if america thinks that the former allies it has screwed over are gonna come running to it's rescue, it has another thought coming. at this point, the only allies america has right now are israel and maybe north korea. but that won't be enough to defend


CONTENT_RELEVANCE_SCORE == 1 



...

comment_id,post_id,post_title,post_body,comment_body,content_relevance_score,reasoning_content_relevance_score
str,str,str,str,str,i64,str
"""murbg5q""","""1kxjjli""","""Black Americans go to Palestine to support and face racism from the Palestinians (word is they were called racial slurs too)""","""[https://x.com/LaCienegaBlvdss/status/1913985212410740972?ref\_src=twsrc%5Etfw%7Ctwcamp%5Etweetembed%7Ctwterm%5E1913985212410740972%7Ctwgr%5E%7Ctwcon%5Es1\_&ref\_url=](https://x.com/LaCienegaBlvdss/status/1913985212410740972?ref_src=twsrc%5Etfw%7Ctwcamp%5Etweetembed%7Ctwterm%5E1913985212410740972%7Ctwgr%5E%7Ctwcon%5Es1_&ref_url=) [https://www.tiktok.com/@hebatalks/video/7489954350727433502?\_r=1&\_t=ZP-8vhOaDP5gZq](https://www.tiktok.com/@hebatalks/video/7489954350727433502?_r=1&_t=ZP-8vhOaDP5gZq) I can't even listen to that Palestinian woman speak her justification on all this.. it's extremely cringe. She has quite a bit to say about ""You can't assume the Palestinians were being racist to you about such and such issue"" when she herself makes so many assumptions in her almost 10-minute long rant. The hypocrisy is abundant here. Not to mention, if these black Americans faced so much racism the whole time, it's becomes very clear that they are indeed being treated with racism when fur…","""Yeah, it is a MAJOR problem in the Middle East. Not only in the Palestinian areas, but also in many other countries in the area, that simply hate Africans. The Houthi rebels, who the pro-Palestinians LOOVE so much, reintroduced slavery... I haven't seen ONE, not ONE of these organizations say anything bad about that... And you probably remember the famous late Eldridge Cleaver, the minister of information for the Black Panther Party and what he said about Algeria... He couldn't believe the level of racism he saw in Algeria and he talked about how the Algerians literally had African slaves. After that he became a Zionist and fierce defender of Israel... He made many controversial anti-Arab remarks, for example he called Arabs the ""most racist people on Earth...""""",3,"""The comment discusses racism in the Middle East, particularly in relation to Palestinians and their treatment of Black Americans, which indirectly touches on the broader context of the Gaza conflict and public opinion. However, it primarily focuses on racial issues rather than the core conflict itself."""
"""nwxt7np""","""1pwh9pg""","""We Hung On By a Thread for Two Years, Now Friendship Is Over""","""Devastated right now. I have a friend who has always been really important to me. We've been friends for a quarter century. We are both half Jewish and have always had that identity in common. He is super lefty and has always been anti-Zionist but not obsessively. Not until two years ago. He started going cold on me when I tried to talk to him about an antisemitic incident that happened at his alma mater. Since then we have remained friends on social media and I reach out to him at Christmas and get a perfunctory response. This time I asked him whether he wants to salvage the friendship. He told me that he thinks Gaza is the holocaust of our time and he is afraid to know what my views are. That's why he hasn't talked to me. Basically he was performing the purity test on me. So I told him my views and reminded him of the way I stood up for Muslims/Arabs who were being harassed and threatened after 9/11. (I literally stood up and faced a couple of thugs who were threatening someone. H…","""This is literally insane. I’m so sorry. I feel I have gotten this same treatment too. How dare we care about Jewish people. It’s like you’re a person of European diaspora who has ancestors who did horrific things but somehow you get a statute of limitations for that. If you are Jewish you are held to another standard and there are tons of different spaces that the justice isn’t held""",3,"""The comment touches on the treatment of Jewish individuals in discussions about the Gaza conflict, reflecting 


CONTENT_RELEVANCE_SCORE == 4 



comment_id,post_id,post_title,post_body,comment_body,content_relevance_score,reasoning_content_relevance_score
str,str,str,str,str,i64,str
"""n9w97kn""","""1mwbg7i""","""Netanyahu: Israel will conquer Gaza regardless of whether Hamas accepts hostage deal""","""Welp, i guess the decision is made then.....""","""It only took them 2 years to do something that should have been done since day 1 🙄 Now let's see if they'll actually follow through instead of just talking""",4,"""The comment expresses skepticism about Israel's actions regarding Gaza, indicating a direct engagement with the ongoing conflict and its implications, which aligns with the core issues at hand."""
"""mq0qal3""","""1kbq23d""","""best way for american to get money to a gaza refugee?""","""hello everyone. myself and a group of people are trying to send the donations we have accumulated to a family in gaza, but we are running into several problems. The money from the gofundme is saved in a separate bank account specifically for Amal, that way it never gets mixed up. We usually wire the money through Western Union, and cash pick up is the only option available. While that had its drawbacks (huge ""service"" costs and a cap from Israhell allowing only 7500 to be sent to one person in Palestine per month, along with it only being open once a week) it DID work. That office that Amal and Awadallah had to physically travel to, which is incredibly dangerous, is now...closed. For obvious reasons. They can no longer access the money through those means. The US does not allow for a direct send to the Bank of Palestine as it is a Zionist country. Many countries in Europe are the same, though there are some that do like the Netherlands. So no direct bank transfers. If there…","""thank you so much for doing this, you can send it direct to Bank of Palestine in case they have a bank account. also please check your DM!""",4,"""The comment provides a suggestion for sending money directly to the Bank of Palestine, which is relevant to the humanitarian efforts related to the Gaza conflict. It addresses the logistical challenges of supporting individuals in Gaza, making it clearly related to the core issues at hand."""
"""ng8etbm""","""1nqbc6v""","""Flotilla = Hamas""","""The Israeli government is holding a press conference right now calling the Flotilla Hamas. I hope the Spanish and Italian naval ships get there quickly. ""","""Anyone who rejects their proposal is Hamas in their eyes. I suspect that it is a projection because Israel buys, guilts, or backmails anyone who still supports Israel at this stage. Israel simply can't understand a group of people coming to the aid of people and bringing attention to a genocide without ulterior motives.""",4,"""The comment critiques Israel's perception of those who oppose its actions and discusses the humanitarian aspect of the conflict, which relates directly to the core issues surrounding the Gaza conflict."""
"""noglo8h""","""1ov0tur""","""Why Even Critical Jews Still Support Israel""","""It’s been hard lately to be a Jew or Israeli and watch what’s happening in Gaza: the destruction, the deaths, the unbearable images. You can’t ignore it, and you shouldn’t. It makes you question everything: your country, your values, your own sense of morality. If an Israeli really dives deep into the history and social dynamics that brought us here, nearly a century of conflict - you’ll find a lot to criticize in our own leadership: arrogance, short-sightedness, moral compromises that have led to oppression and displacement. And yet, even the most critical and informed Israelis often still find themselves on Israel’s side. There's a reason for this. There are a few hard, uncomfortable truths behind that. *1. The counterfactual matters.* If Israel had never been created, there probably wouldn’t be a Palestinian state either. The land would likely have been absorbed into neighboring powers - maybe Syria, maybe Jordan - and its people would have lived under


CONTENT_RELEVANCE_SCORE == 5 



comment_id,post_id,post_title,post_body,comment_body,content_relevance_score,reasoning_content_relevance_score
str,str,str,str,str,i64,str
"""nbek5lh""","""1n33b22""","""Israeli Army Recovers Body of Ilan Weiss, Kidnapped to Gaza on October 7""","""""","""I have a feeling soon the rest of the hostages will be found, dont know if anymore will be alive but we can pray that some will make it out alive""",5,"""The comment expresses concern about the fate of hostages taken to Gaza, which is directly related to the ongoing conflict and its humanitarian implications."""
"""mpk6msk""","""1ka46zf""","""I'm so fucking angry""","""I followed a link from another post on this sub to read a couple articles on a zionist groups' assault on an antizionist Jew in NYC, along with a video of zionists chanting ""death to arabs"" and ""fuck fuck palestine"" (I admit that I couldn't tell what they were saying from the audio but that is what the captions said. I guess it's possible the captions could be inaccurate but I'm not sure they are). Just fucking IMAGINE if a pro-israel Jew was assaulted by a crowd of people screaming 'death to jews'. Hell, I bet a headline like that has already been fabricated by some zionist out there. Zionism in the name of safety is anti-safety. For EVERYONE. It is anti Israeli safety, anti Jewish safety, anti childrens safety, anti journalistic safety, anti free speech. I am so fucking SICK of the lies and the bullshit. These people - many of whom otherwise claim to be lefties - are guzzling the dick of fascism because of their blind-ass stance on this one issue. They have completely lost the…","""They most certainly DID say ""death to Arabs"". ""Ma'avet La'aravim"" is death to Arabs, and this Zionist was trying to gaslight me into believing they were chanting ""truth for the Jews"" which is ""Emet La'yehudim"". I don't know how that's even an argument floating around, the two aren't easy to confuse even for non-Hebrew speakers""",5,"""The comment directly addresses the chanting of anti-Arab slogans and the implications of Zionism, which are relevant to the ongoing conflict and public sentiment surrounding it. This discussion of specific incidents and their broader societal impact ties closely to the core issues of the Gaza conflict."""
"""n7enm49""","""1mjtc9u""","""Tired of this anti-Gazan hivemind rhetoric - all people are individuals!""","""Time to speak out about the Anti-Gazan rhetoric and continue to remind people that there are all types of people in Gaza!!! Yes, humans actually have **nuance** and **individuality**, surprising! Yes, sci-fi is very fun to read, but Gazans are not capable of hivemind thinking, just like the rest of us humans, although that would be *really cool!* >**Hive mind:** the collective mental activity expressed in the complex, coordinated behavior of a colony of social insects (such as bees or ants) regarded as comparable to a single mind controlling the behavior of an individual organism [https://www.merriam-webster.com/dictionary/hive%20mind](https://www.merriam-webster.com/dictionary/hive%20mind) If you don't like being caricaturized, then please don't do it to 2m+ people! This is a common tactic throughout history to justify collective punishment, which is also *illegal*! Although apparently *international law does not apply in Gaza!* International law posits that no protected person ma…","""This is correct except for 2 things: \- Hamas since being elected in 2006 is the political party that hasn't changed in 19 years. \- Israeli Prime Minister Naftali Bennet has indicated that Hamas has 76/132 seats in Palestine Parliament or over 50%, [https://abcnews.go.com/International/former-israeli-prime-minister-naftali-bennett-vies-position/story?id=113345773](https://abcnews.go.com/International/former-israeli-prime-minister-naftali-bennett-vies-position/story?id=113345773)""",5,"""The comment addresses the rhetoric surrounding Gazans and collective punishment, discussing the implications of 

---

In [26]:
import json
script_path = os.getcwd()
project_path = os.path.join(script_path, '..', '..')
labeling_dir = os.path.join(project_path, 'data', 'labeled_samples')
val_sample_path = os.path.join(labeling_dir, '04a_validation_sample.json')
with open(val_sample_path, "r", encoding="utf-8") as f:
    val_sample = json.load(f)

In [28]:
comment_ids = [x['comment_id'] for x in val_sample]

In [31]:
processed_df_filtered.filter(
    pl.col('comment_id').is_in(comment_ids),
    pl.col('content_relevance_score') == 4
    )

comment_id,post_id,post_title,post_body,comment_body,content_relevance_score,reasoning_content_relevance_score
str,str,str,str,str,i64,str
"""mtwl83s""","""1kt11kd""","""Dana Bash and guest: “Free Palestine” means violence against Jews""","""""","""Hahaha I caught the sarcasm you actually mean free Palestine is free Palestine""",4,"""Primary subject: interpretation of the 'Free Palestine' slogan in response to a media claim; this is a brief reaction about protest messaging and public opinion related to the Gaza/Palestine issue, so it fits the 'media/opinion/protests' category."""
"""nd24kod""","""1nbhq9c""","""[MEGATHREAD] Numerous injured and dead reported in major gunfire terror attack near Jerusalem""","""As of 14:23 am Israeli time, there are reports of 6 dead and 10+ injured. The scene is still on-going and getting udpated. Hebrew sources: [Mako](https://www.mako.co.il/news-military/2025_q3/Article-f7e0d9d43282991027.htm) [Ynet](https://www.ynet.co.il/news/article/hjclqzhcel) [Kan](https://www.kan.org.il/content/kan-news/defense/949194/) English sources: [Times of Israel](https://www.timesofisrael.com/liveblog_entry/shooting-reported-at-ramot-junction-entrance-to-jerusalem/)""","""I'm not from Israel, but I just came to say that this is heartbreaking. My condolences to the victims' friends and families.""",4,"""The comment expresses sympathy for the victims of a violent incident related to the conflict, which reflects a personal reaction to the ongoing situation in the region."""
"""oaelbxl""","""1rszuct""","""""Israelis"" in the West Bank attack Palestinian fast food workers, who make their food, for being Arabs""","""""","""complete sickos """,4,"""The comment expresses a negative sentiment towards the actions of Israelis in the West Bank, which relates to the broader context of the Gaza conflict and the treatment of Palestinians."""
"""o9gyifr""","""1rovyj0""","""Why is Iran bombing civilian targets in friendly countries?""","""Since the war started, Iran has been using missiles and drones against civilian targets in neighboring countries. They've hit a hotel in Bahrain, a residential neighborhood in Saudi Arabia, the airport in Dubai, and others. Even desalinization plants, which some countries depend on for survival. I can see why they'd shoot at Israel or at American bases in Gulf countries. But why so many civilian targets in countries that are not involved in the war and are the closest thing Iran has to allies? ""","""The oil refineries are an economic tool. The airport in Dubai hosts USAF satellite infrastructure and C&C. Some US military personnel has moved from their bases into civilian buildings like the apartment building that got droned which hosted fleet command personnel and the mainstream media is speculating that certain other buildings hosted intelligence operations However the over arching idea is to make this war as economically costly to the gulf states, Israel and US that any victory ends up being pyrrhic""",4,"""The comment discusses Iran's military actions and their implications for the Gulf states and Israel, which are directly related to the broader context of the Gaza conflict and its regional impact."""
"""nb5u7op""","""1n0ppk8""","""Poll: 41% of Americans say 'Israel IS committing Genocidal Acts in Gaza'""","""""","""The other 59% are willfully ignorant or complicit""",4,"""The comment addresses public opinion regarding Israel's actions in Gaza, directly relating to the ongoing conflict and the perception of genocide, which is a core aspect of the discussion."""
…,…,…,…,…,…,…
"""n3o27ca""","""1m2chzb""","""Nobody really cares about genocide.""","""Nobody really cares about the what is happening in Gaza. If anyone really cared about genocide, they would also care about the starving children in Africa who are dying by the tens of millions, about the hundreds of thousands killed in the Sudanese war, and a host of other bloody conflicts that the world has ignored, because it simply didn't care! 

In [34]:
processed_df_filtered.filter(
    pl.col('comment_id').is_in(comment_ids),
    pl.col('content_relevance_score') == 5
    )['comment_id'].to_list()

['mz8ans8',
 'mtrkdzf',
 'njdvcit',
 'nsz6783',
 'n27i0kt',
 'nfgcrg1',
 'my7v96k',
 'nvpu8hc',
 'mziz50p',
 'nitry3y',
 'ntck9ht',
 'n9fqfdz']